In [1]:
import random
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import re

from keras.layers import TextVectorization,Embedding,Dense,SimpleRNN,Dropout,Lambda
from keras.models import Sequential
from keras.optimizers import Adam

from sklearn.model_selection import train_test_split

2024-01-06 23:23:06.138853: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [3]:
df = pd.read_csv('imdb.csv')
df = df.sample(10000)

In [4]:
stop_words = ['is','a','the','he','she']

In [5]:
def clean(text):
    result = text.lower()
    result = re.sub(r'<[^>]*>', r' ', result)
    result = re.sub(r'[^a-z\s\']', r'', result)
    result = re.sub(r'\b(?:' + '|'.join(stop_words) + r')\b', ' ', result)
    result = re.sub(r'\s+', r' ', result)
    return result

df['Cleaned'] = df['Review'].apply(clean)

In [6]:
x_train, x_test, y_train, y_test = train_test_split(df['Cleaned'],df['Sentiment'],test_size=.2,random_state=seed)

In [7]:
max_tokens = 1500
output_sequence_length = 500

vectorizer = TextVectorization(max_tokens=max_tokens,output_sequence_length=output_sequence_length)
vectorizer.adapt(x_train.values)

In [8]:
model = Sequential([
    vectorizer,
    Embedding(input_dim=max_tokens, input_length=output_sequence_length, output_dim=256, mask_zero=True),
    SimpleRNN(4),
    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_vectorization (TextVe  (None, 500)               0         
 ctorization)                                                    
                                                                 
 embedding (Embedding)       (None, 500, 256)          256000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 4)                 1044      
                                                                 
 dense (Dense)               (None, 1)                 5         
                                                                 
Total params: 257049 (1004.10 KB)
Trainable params: 257049 (1004.10 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [9]:
optimizer = Adam(learning_rate=.0001)
model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
history = model.fit(x_train,y_train,epochs=25,batch_size=25,validation_split=.2).history

Epoch 1/25
256/256 [==============================] - 29s 107ms/step - loss: 0.6924 - accuracy: 0.5213 - val_loss: 0.6890 - val_accuracy: 0.5519
Epoch 2/25
 48/256 [====>.........................] - ETA: 20s - loss: 0.6688 - accuracy: 0.6700

In [ ]:
model.evaluate(x_test,y_test)

In [ ]:
plt.plot(history['loss'])
plt.plot(history['val_loss'])
plt.show()

In [ ]:
plt.plot(history['accuracy'])
plt.plot(history['val_accuracy'])
plt.show()

In [ ]:
predictions = model.predict(x_test)
testing = pd.DataFrame(y_test.values,index=y_test.index, columns=['Actual'])
testing['Prediction'] = predictions
testing['Rounded'] = testing['Prediction'].round().astype(int)

In [ ]:
pd.crosstab(testing['Actual'],  testing['Rounded'], rownames=['Actual'], colnames=['Predicted'])

In [ ]:
plt.scatter(y_test,predictions)
plt.xlim(0,1)
plt.ylim(0,1)
plt.show()